# Exploratory Data Analysis - Package Forecasting

This notebook explores the historical package data for locations A, B, and C.

In [118]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.stattools import adfuller
import numpy as np
import holidays

## 1. Load Data

In [119]:
# Load raw data
df = pd.read_csv("../data-4-.csv")
df["date"] = pd.to_datetime(df["date"])

# Convert location columns to numeric
for col in ["location_A", "location_B", "location_C"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Data shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df = df.iloc[:-30]  # "hide" the test data when doing EDA to avoid data leakage

# Melt the DataFrame to long format for easier plotting
df_long = df.melt(
    id_vars=["date"],
    value_vars=["location_A", "location_B", "location_C"],
    var_name="Location",
    value_name="Volume",
)

df_long.head()

Data shape: (1338, 4)
Date range: 2022-01-02 00:00:00 to 2025-08-31 00:00:00


,date,Location,Volume
0,2022-01-02,location_A,8591.0
1,2022-01-03,location_A,11363.0
2,2022-01-04,location_A,12196.0
3,2022-01-05,location_A,11487.0
4,2022-01-06,location_A,6860.0


## 2. Missing Data Analysis

In [120]:
# Check missing values
missing = df.isnull().sum()
print("Missing values:")
print(missing)
print(f"\nMissing percentages:")
print((missing / len(df) * 100).round(2))

Missing values:
date            0
location_A      0
location_B     14
location_C    973
dtype: int64

Missing percentages:
date           0.00
location_A     0.00
location_B     1.07
location_C    74.39
dtype: float64


## 3. Time Series Visualization

In [121]:
fig = px.line(
    df_long,
    x="date",
    y="Volume",
    facet_row="Location",
    color="Location",
    title="Volume Trends by Location",
    template="plotly_white",
    height=600,
)

fig.update_layout(showlegend=False)

fig.show()

## 4. Statistical Summary

In [122]:
# Summary statistics
summary = df[["location_A", "location_B", "location_C"]].describe()
print("Summary statistics:")
print(summary)

Summary statistics:
         location_A    location_B    location_C
count   1308.000000   1294.000000    335.000000
mean   13614.313456   5515.291345   4295.220896
std     7097.501583   3816.939937   3171.690965
min        0.000000      0.000000      0.000000
25%     9074.500000   2871.750000      3.000000
50%    13001.000000   4976.500000   4917.000000
75%    17900.000000   8011.250000   6636.500000
max    38136.000000  16882.000000  13180.000000


In [123]:
results = []
for loc in ["location_A", "location_B", "location_C"]:
    series = df_long[df_long["Location"] == loc]["Volume"].dropna()
    adf_result = adfuller(series)

    results.append(
        {
            "Location": loc,
            "ADF Statistic": adf_result[0],
            "p-value": adf_result[1],
            "Stationary": "Yes" if adf_result[1] < 0.05 else "No",
        }
    )

# Display as a clean DataFrame
adf_df = pd.DataFrame(results)
print(adf_df)

     Location  ADF Statistic   p-value Stationary
0  location_A      -2.534101  0.107396         No
1  location_B      -1.876538  0.343178         No
2  location_C      -2.368290  0.150887         No


## 5. Seasonality Check

In [124]:
# Extracting time components
df_long["Day_of_Week"] = df_long["date"].dt.day_name()
df_long["Month"] = df_long["date"].dt.month_name()
df_long["Year"] = df_long["date"].dt.year


# WEEKLY SEASONALITY

# Define the sort order for days of the week
days_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

fig_weekly = px.box(
    df_long,
    x="Day_of_Week",
    y="Volume",
    color="Location",
    facet_col="Location",
    category_orders={"Day_of_Week": days_order},
    title="Weekly Seasonality: Volume Distribution by Day of Week",
    template="plotly_white",
)
fig_weekly.update_layout(showlegend=False)
fig_weekly.show()

# MONTHLY SEASONALITY

fig_monthly = px.box(
    df_long,
    x="Month",
    y="Volume",
    color="Location",
    facet_col="Location",
    title="Monthly Seasonality: Volume Distribution by Month",
    template="plotly_white",
)
fig_monthly.update_layout(showlegend=False)
fig_monthly.show()

# YEARLY SEASONALITY
fig_yearly = px.box(
    df_long,
    x="Year",
    y="Volume",
    color="Location",
    facet_col="Location",
    title="Yearly Seasonality: Volume Distribution by Year",
    template="plotly_white",
)
fig_yearly.update_layout(showlegend=False)
fig_yearly.show()

In [ ]:
# Setup Swedish Holidays
se_holidays = holidays.SE()

df_long["is_holiday"] = df_long["date"].apply(lambda x: x in se_holidays)

fig_holiday = px.box(
    df_long,
    x="is_holiday",
    y="Volume",
    color="Location",
    facet_col="Location",
    title="Impact of Holidays on Volume",
    template="plotly_white",
)

fig_holiday.update_layout(showlegend=False)
fig_holiday.show()

In [126]:
locations = ["location_A", "location_B", "location_C"]
fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=[f"ACF: {loc}" for loc in locations],
    vertical_spacing=0.1,
    shared_xaxes=True,
)

for i, loc in enumerate(locations):
    # 1. Isolate the series
    series = df_long[df_long["Location"] == loc]["Volume"].dropna()

    # 2. Calculate ACF and Confidence Intervals
    n_obs = len(series)
    acf_values, conf_int = acf(series, nlags=30, alpha=0.05)
    lags = np.arange(len(acf_values))

    # Standard Error for ACF is approximately 1/sqrt(N)
    ci_bound = 1.96 / np.sqrt(n_obs)

    # 3. Add the ACF
    fig.add_trace(go.Bar(x=lags, y=acf_values, name=loc), row=i + 1, col=1)

    # Any bar poking OUT of this box is a real seasonal or trend pattern
    fig.add_shape(
        type="rect",
        x0=0,
        x1=max(lags),
        y0=-ci_bound,
        y1=ci_bound,
        fillcolor="rgba(0,100,80,0.2)",
        line_width=0,
        row=i + 1,
        col=1,
    )

# Formatting
fig.update_layout(
    height=900,
    title_text="Corrected Autocorrelation (ACF) by Location",
    showlegend=False,
    template="plotly_white",
)
fig.update_xaxes(title_text="Lags (Days)", row=3, col=1)
fig.update_yaxes(title_text="Correlation", range=[-0.1, 1.1])

fig.show()

## Key Findings

1. Location C only has data starting from around September 2024, while A and B have data from January 2022.

2. **Weekly Seasonality**: All locations show clear weekly and holiday patterns (lower volumes on weekends and holidays).

3. **Volume Differences**: Locations have different scales.

